# Retail Multi-Store Hierarchical Demand Forecaster - MLOps Pipeline

Welcome to the end-to-end MLOps pipeline for Retail Demand Forecasting on the **Kaggle Walmart M5 Dataset**.

**Architecture Overview:**
- **Primary Code Source:** Modular Python scripts under `src/` (`src/utils.py`, `src/data_ingestion.py`, `src/eda.py`, `src/preprocessing.py`, etc.).
- **Secondary Interactive Notebook:** Imports and executes `src/` modules stage-by-stage with visual outputs and interactive explanations.

---
## Phase 1: Environment Setup & Data Pipeline Ingestion

In this stage, we initialize the project directories and download the real Walmart M5 dataset (30,490 time series over 1,913 days across 10 stores in CA, TX, WI).

In [ ]:
# 1.1 Environment Setup using src.utils
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.utils import set_seed, init_project_dirs
from src.data_ingestion import run_data_ingestion

set_seed(42)
dirs = init_project_dirs(base_dir=project_root)

# Ingest real Kaggle M5 dataset
df_sales, df_calendar, df_prices, metrics = run_data_ingestion(
    raw_data_dir=os.path.join(project_root, 'data', 'raw'),
    use_real_data=True
)

---
## Phase 1.5: Complete 11-Part Exploratory & Multi-Attribute Grid Analysis

We analyze the data through 11 comprehensive diagnostic modules designed to extract all variance-driving signals for our forecasting model:
1. **Univariate Analysis:** Zero-inflation distribution & category price dispersion.
2. **Bivariate Analysis:** Store/State sales velocity & SNAP policy lifts.
3. **Multivariate & Time Series Decomposition:** 5-year macro trend & Autocorrelation (ACF/PACF).
4. **Composite Price Elasticity & Markdowns:** Sales multiplier vs. discount depth tiers.
5. **Composite Payday & Cashflow Cycles:** Bimonthly paycheck surges & multi-state SNAP concurrency.
6. **Syntetos-Boylan Demand Categorization:** ADI vs. CV² profiling (Smooth / Intermittent / Erratic / Lumpy).
7. **Department Basket Mix Share:** Monthly wallet-share rotations across seasons.
8. **10-Store Facet Grid:** Cross-store and cross-category heterogeneity.
9. **Event Shock Spectrum & 14-Day Ramp Grid:** Holiday lead-up ramp curves & event type sensitivities.
10. **Multi-Lag Autocorrelation Grid:** Lag correlation spectrum (Lags 7 to 364 days).
11. **Zero-Streak & Item Lifecycle Grid:** Shelf dwell times & new item 60-day launch adoption curves.

In [ ]:
# 1.5.1 Foundations: Univariate, Bivariate & Multivariate Analysis
from src.eda import univariate_analysis, bivariate_analysis, multivariate_time_series_analysis

fig_dir = os.path.join(project_root, 'reports', 'figures')
univariate_analysis(df_sales, df_prices, output_dir=fig_dir)
bivariate_analysis(df_sales, df_calendar, df_prices, output_dir=fig_dir)
multivariate_time_series_analysis(df_sales, df_calendar, output_dir=fig_dir)

In [ ]:
# 1.5.2 Composite Trends: Price Elasticity, Paydays & Syntetos-Boylan Profiling
from src.eda import (
    composite_price_elasticity_analysis,
    composite_calendar_payday_analysis,
    composite_demand_classification_grid,
    composite_department_basket_mix
)

composite_price_elasticity_analysis(df_sales, df_calendar, df_prices, output_dir=fig_dir)
composite_calendar_payday_analysis(df_sales, df_calendar, output_dir=fig_dir)
composite_demand_classification_grid(df_sales, output_dir=fig_dir)
composite_department_basket_mix(df_sales, df_calendar, output_dir=fig_dir)

In [ ]:
# 1.5.3 Multi-Attribute Variance Grids: Stores, Events, Multi-Lags & Lifecycles
from src.eda import (
    grid_store_department_heterogeneity,
    grid_event_type_shock_spectrum,
    grid_lag_correlation_variance,
    grid_zero_streaks_item_lifecycle
)

grid_store_department_heterogeneity(df_sales, df_calendar, output_dir=fig_dir)
grid_event_type_shock_spectrum(df_sales, df_calendar, output_dir=fig_dir)
grid_lag_correlation_variance(df_sales, output_dir=fig_dir)
grid_zero_streaks_item_lifecycle(df_sales, df_calendar, df_prices, output_dir=fig_dir)

---
## Phase 2: Data Preprocessing & Memory Optimization

In this stage, we execute `src.preprocessing.melt_and_merge_data()`:
1. Transform 30,490 wide-format sales series (`d_1` ... `d_1913`) into long panel records.
2. Merge calendar event tags, SNAP benefit indicators, and weekly price movements.
3. Apply our custom memory downcasting engine (`reduce_mem_usage`) to reduce RAM consumption by ~80%.
4. Export the resulting 25.76M-row master panel dataset to `data/processed/grid_part_1.parquet`.

In [ ]:
# 2.1 Execute Phase 2 Preprocessing & Memory Optimization
from src.preprocessing import melt_and_merge_data

df_grid = melt_and_merge_data(
    raw_data_dir=os.path.join(project_root, 'data', 'raw'),
    processed_data_dir=os.path.join(project_root, 'data', 'processed'),
    start_day=1069  # Most recent ~2.5 years of high-relevance data
)

print('\n📊 Master Processed Grid Sample:')
df_grid.head(3)